# OpenAI API Exploration

Hands-on walkthrough of the OpenAI Python SDK and LangChain's OpenAI integration — for GenAI/LLM interview prep.

**Covers:**
- Environment setup and API key management
- Chat Completions API (`client.chat.completions.create`)
- Responses API (OpenAI's newer API surface)
- LangChain's `ChatOpenAI` wrapper — messages, prompt templates, conversation history
- Tool / function calling
- Structured output with Pydantic

In [27]:
!uv add python-dotenv
!uv add langchain-openai

Resolved 107 packages in 4ms
Checked 101 packages in 1ms
Resolved 122 packages in 2.70s                                       
Prepared 13 packages in 4.49s                                            
Installed 15 packages in 77ms                               
 + distro==1.9.0
 + jsonpatch==1.33
 + langchain-core==1.6.1
 + langchain-openai==1.6.0
 + langchain-protocol==0.0.19
 + langsmith==0.12.1
 + orjson==3.12.0
 + regex==2026.9.3
 + requests-toolbelt==1.0.0
 + tenacity==9.1.4
 + tiktoken==0.14.0
 + uuid-utils==0.17.0
 + websockets==17.1
 + xxhash==4.0.1
 + zstandard==0.25.0


## Setup

Install dependencies, import the SDK, and load the API key from `.env` via `python-dotenv`. Keeping secrets out of source/notebook cells (and out of git) is standard practice — a common interview question is "how do you manage API keys/secrets in an app?"

In [17]:
import openai

In [18]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

## Chat Completions API

`client.chat.completions.create(...)` is OpenAI's original, still widely-used API for chat-style interactions.

- `model` — which model to call
- `messages` — a list of `{"role": ..., "content": ...}` dicts (`system` sets behavior, `user`/`assistant` are the turns)
- `temperature` — 0 = deterministic/focused, higher = more random/creative

Interview angle: be ready to contrast this with the newer **Responses API** below.

In [20]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello and tell me a fun fact."}
    ],
    temperature=0.7,
)

print(response.choices[0].message.content)

Hello! Here’s a fun fact: Honey never spoils! Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible. This is due to honey's low moisture content and acidic pH, which create an inhospitable environment for bacteria and microorganisms.


## Responses API

`client.responses.create(...)` is OpenAI's newer API, designed to unify chat + tool use + multi-turn state handling. Key differences from Chat Completions:
- `input` can be a plain string *or* a list of role-based messages
- roles include `developer` (replaces `system`) and `user`
- an `instructions` parameter is a shortcut for a system/developer message
- supports server-side conversation state via `previous_response_id` (no need to resend full history yourself)

**⚠️ Note on the model's answer below:** it's wrong — the model hallucinates, describing the Python `responses` HTTP-mocking library and generic REST "response wrapper" patterns instead of OpenAI's actual Responses API. This is a nice real example to bring to an interview: LLMs confidently answer about their *own* newer APIs incorrectly when the API postdates their training data — always verify against docs rather than trusting a self-referential answer.

In [21]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="Does create API is only used for ?"
)

print(response.output_text)

The "create" API is typically used for generating new resources within a system. Its primary functions often include:

1. **Creating New Records**: This could involve adding new entries to a database, like users, products, or transactions.

2. **Uploading Files**: Some APIs allow users to upload documents or images.

3. **Initiating Processes**: This could include starting workflows or processes in applications.

4. **Configuring Settings**: Some APIs allow the creation of new settings or configurations within an application.

The specific use can vary based on the type of API and its purpose, but generally, the "create" method focuses on generating new data or instances. If you have a specific context in mind, I can provide more tailored information!


**Note:** the `"developer"` role here is the Responses API's equivalent of the old `"system"` role — same purpose (steering instructions), new name.

In [22]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "developer",
            "content": "You are an expert Python instructor."
        },
        {
            "role": "user",
            "content": "What is the Responses API used for?"
        }
    ]
)

print(response.output_text)

The Responses API is typically used in the context of web development and testing, particularly in the field of Python programming. It allows developers to mock and stub HTTP responses in their tests, which is useful for isolating and testing code that interacts with external APIs without actually hitting the external service.

### Key Features and Uses of the Responses API:

1. **Mocking HTTP Responses**:
   - Instead of making real HTTP requests, you can specify the expected request and define what response the API should return. This helps in testing error handling without relying on the actual endpoints.

2. **Testing Edge Cases**:
   - You can simulate various scenarios, including successful responses, timeouts, and HTTP errors (like 404 or 500), making it easier to ensure your code handles them correctly.

3. **Improved Test Performance**:
   - Mocking API responses speeds up code execution during tests since there's no network latency involved in real HTTP requests.

4. **Isolat

**Shortcut:** instead of a `developer`-role message, pass `instructions="..."` as its own top-level parameter — functionally equivalent, less boilerplate for the single-instruction case.

In [25]:
response = client.responses.create(
    model="gpt-4o-mini",

    instructions="You are an expert Python instructor.",

    input="What is the Responses API used for?"
)
print(response.output_text)

The Responses API is typically used in various frameworks and libraries for managing API responses in web applications. While the specifics can vary depending on the context (like whether you’re in a web framework like Flask or Django, or working with external services), here’s a general overview of the key purposes and features of a Responses API:

### Key Purposes

1. **Handling API Responses**:
   - Facilitate consistent responses from server APIs, providing a streamlined way to format data sent back to the client.

2. **Data Formatting**:
   - Standardizes the structure of responses, often including status codes, messages, and payloads in a unified format (like JSON or XML).

3. **Error Management**:
   - Provides a method for encapsulating error handling, allowing developers to send meaningful error messages and codes back to clients.

4. **Testing and Mocking**:
   - In testing environments, responses APIs can be used to mock responses from external services, enabling testing of 

## LangChain: `ChatOpenAI`

LangChain wraps OpenAI (and other providers) behind a common `ChatOpenAI` interface, so the same code can target different LLM providers with minimal changes. `.invoke(...)` accepts either a plain string or a list of message objects.

In [28]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

response = llm.invoke("Explain Kubernetes in simple terms")

print(response.content)

Kubernetes is like a manager for your applications that run in containers. Here's a simple breakdown:

1. **Containers**: Think of containers as small, lightweight packages that hold everything an application needs to run (like its code, libraries, and settings). They help ensure the application runs the same way no matter where it's deployed.

2. **Orchestration**: This is where Kubernetes comes in. It helps you manage lots of these containers, making sure they run smoothly, are deployed correctly, and can easily scale up or down based on demand.

3. **Automation**: Kubernetes automates many tasks, like starting and stopping containers, checking their health, and scaling them based on traffic. This means you don’t have to do everything manually.

4. **Scaling**: If your application suddenly needs to handle more users, Kubernetes can automatically add more containers to manage that load.

5. **Self-Healing**: If something goes wrong with a container, Kubernetes can restart it or replac

### Structured message objects

Instead of a raw string, pass `SystemMessage` / `HumanMessage` objects for explicit role control — LangChain's equivalent of the `role`/`content` dicts used directly against the OpenAI API.

In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(
        content="You are an expert Kubernetes instructor."
    ),
    HumanMessage(
        content="Explain Kubernetes Pods."
    )
]

response = llm.invoke(messages)

print(response.content)

Kubernetes Pods are the smallest deployable units in the Kubernetes ecosystem and serve as the fundamental building blocks for running applications in a Kubernetes cluster. They encapsulate one or more containers, along with shared storage, networking, and configuration options for those containers.

### Key Characteristics of Pods:

1. **Multiple Containers**: A Pod can host one or multiple containers. Containers in the same Pod share the same network namespace, meaning they can communicate with each other via `localhost` and can share storage volumes. This is particularly useful for closely related applications that need to work together, like a web server and a logging agent.

2. **Shared Resources**: All containers within a Pod share resources such as networking and storage. This implies they will have the same IP address, port space, and can communicate with each other over the localhost interface.

3. **Lifecycle Management**: Pods are designed to be transient. They can be create

### Multi-turn conversation history

Include prior `AIMessage` turns in the list to give the model context from earlier in the conversation. This is how chat "memory" actually works — you resend the full message history on every call; there's no server-side memory unless you add it yourself (e.g. LangGraph checkpointing — see `langgraph.ipynb`).

In [30]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

llm = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(
        content="You are a Python teacher."
    ),

    HumanMessage(
        content="What is a list?"
    ),

    AIMessage(
        content="A list is an ordered collection of objects."
    ),

    HumanMessage(
        content="How is it different from a tuple?"
    )
]

response = llm.invoke(messages)

print(response.content)

Lists and tuples are both data structures in Python that can store collections of items, but they have some key differences:

1. **Mutability**:
   - **List**: Lists are mutable, which means you can modify them after they have been created. You can add, remove, or change items in a list.
   - **Tuple**: Tuples are immutable, meaning once they are created, their contents cannot be changed. You cannot add, remove, or modify items in a tuple.

2. **Syntax**:
   - **List**: Lists are defined using square brackets `[]`. For example: `my_list = [1, 2, 3]`.
   - **Tuple**: Tuples are defined using parentheses `()`. For example: `my_tuple = (1, 2, 3)`.

3. **Performance**:
   - **List**: Lists typically have a slightly higher performance overhead than tuples due to their mutability.
   - **Tuple**: Tuples can be more memory efficient than lists and can be faster for certain operations because they are immutable.

4. **Use Cases**:
   - **List**: Files and data that may need to be modified (e.g

## Prompt Templates

`ChatPromptTemplate` parameterizes prompts with `{placeholders}` and composes with a model using the `|` (pipe) operator — LangChain Expression Language (LCEL): `chain = prompt | llm`. `chain.invoke({...})` fills the template and runs it in one call.

In [31]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert {domain} instructor."
    ),
    (
        "human",
        "Explain {topic} in simple terms."
    )
])

chain = prompt | llm

response = chain.invoke({
    "domain": "Kubernetes",
    "topic": "StatefulSets"
})

print(response.content)

Sure! A StatefulSet is a Kubernetes resource used to manage stateful applications, which are applications that require stable and unique network identifiers and stable storage. 

Here's a breakdown of its key features in simple terms:

1. **Stable Identity**: Each pod in a StatefulSet gets a unique identifier that stays the same even if the pod is restarted. This identity is important for applications that need to maintain a connection to other services.

2. **Stable Storage**: StatefulSets can automatically provision persistent storage by associating each pod with a persistent volume. This means that even if a pod is deleted or rescheduled, it can get the same storage back.

3. **Orderly Deployment and Scaling**: When you create or scale a StatefulSet, Kubernetes ensures that pods are started or stopped in a specific order. For example, if you are scaling up from 2 to 3 pods, Kubernetes will create the 3rd pod only after the 2nd pod is ready.

4. **Ordered Termination**: When scaling 

## Tool / Function Calling

Decorate a Python function with `@tool` to expose it to the model. `llm.bind_tools([...])` tells the model what's available; the model decides *whether* and *how* to call a tool but never executes it — it just returns a `tool_calls` list describing the function name + arguments it wants invoked.

In [34]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is 30°C."

llm = ChatOpenAI(model="gpt-4o-mini")

llm_with_tools = llm.bind_tools([get_weather])

response = llm_with_tools.invoke(
    "What is the weather in Delhi?"
)

print(response.tool_calls)

[{'name': 'get_weather', 'args': {'city': 'Delhi'}, 'id': 'call_G428BjD5i8dTZeWBz4zvhxq4', 'type': 'tool_call'}]


**Note:** your code executes the tool, not the model. `get_weather.invoke(tool_call["args"])` runs the actual function with the arguments the model chose. In a full agent loop, you'd feed this result back to the model as another message so it can produce a final answer — that's exactly the loop built with LangGraph in the other notebook.

In [37]:
tool_call = response.tool_calls[0]

result = get_weather.invoke(
    tool_call["args"]
)

print(result)

The weather in Delhi is 30°C.


## Structured Output

`with_structured_output(PydanticModel)` forces the response to conform to a schema — you get back a validated Python object instead of free text. This is the mechanism behind most LLM-based data-extraction pipelines, and under the hood it's typically implemented via tool/function calling or JSON mode.

In [38]:
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

class Person(BaseModel):
    name: str
    age: int

llm = ChatOpenAI(model="gpt-4o-mini")

structured_llm = llm.with_structured_output(Person)

result = structured_llm.invoke(
    "John is 32 years old."
)

print(result.name)
print(result.age)

John
32


## Recap

- Two OpenAI APIs: **Chat Completions** (classic, `messages` list) vs **Responses** (newer, more flexible input/state handling)
- LangChain's `ChatOpenAI` gives a provider-agnostic interface over any of them
- Prompts can be templated and composed with LCEL (`prompt | llm`)
- **Tool calling**: the model *chooses* a tool + arguments; your code executes it and (in a loop) feeds the result back
- **Structured output**: force schema-conformant responses for reliable downstream parsing